# Chronos-View — M3 IC-Light

Runs the official text-conditioned IC-Light implementation on the Check-in 1 input: `data/crops/shattuck_downtown_2025-03/view_000.png`. Use a Google Colab **GPU** runtime. Do not upload or commit restricted imagery, generated results, weights, credentials, or Hugging Face caches to this repository.

This is an outdoor street-scene stress test. The primary run deliberately calls the official `process` function rather than `process_relight`, so that BRIA's portrait-oriented automatic background removal does not preprocess the complete street scene.

## 1. Enable a GPU and install the official implementation

In Colab select **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU), then run this cell. The official repository downloads model files on first use; it is not copied into Chronos-View.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

subprocess.run(['nvidia-smi'], check=True)
if not Path('/content/IC-Light').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/lllyasviel/IC-Light.git', '/content/IC-Light'], check=True)
os.chdir('/content/IC-Light')
# Install only IC-Light's PyTorch inference dependencies. The original Gradio/protobuf
# UI pins conflict with current Colab TensorFlow and are unused by this notebook.
iclight_packages = [
    'diffusers==0.27.2', 'transformers==4.36.2', 'huggingface-hub==0.20.3',
    'accelerate==0.27.2', 'opencv-python', 'safetensors', 'pillow==10.2.0',
    'einops', 'peft==0.7.1',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', *iclight_packages], check=True)
print('IC-Light commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

import torch
assert torch.cuda.is_available(), 'IC-Light requires a CUDA GPU. Enable a Colab GPU runtime and restart from this cell.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
os.environ['USE_TF'] = '0'
import accelerate, diffusers, transformers
print('accelerate:', accelerate.__version__, '| diffusers:', diffusers.__version__, '| transformers:', transformers.__version__)

## 2. Upload the exact shared crop

Upload only `data/crops/shattuck_downtown_2025-03/view_000.png`, naming it `view_000.png`. The assertions prevent accidentally comparing a different crop or resolution.

In [ ]:
import hashlib
from pathlib import Path
from google.colab import files
from PIL import Image

# Colab writes uploads into the current directory; setup changed it to /content/IC-Light.
INPUT_PATH = Path.cwd() / 'view_000.png'
if not INPUT_PATH.exists():
    uploaded = files.upload()
    if 'view_000.png' not in uploaded:
        raise ValueError('Upload the exact shared crop and keep its filename as view_000.png.')

EXPECTED_SHA256 = '01c4a516920dd6ab02125cae1dd8f0c1ea0f4ef3bce1a61513dce9af890dc483'
actual_sha256 = hashlib.sha256(INPUT_PATH.read_bytes()).hexdigest()
assert actual_sha256 == EXPECTED_SHA256, (
    'This is not the exact M1 view_000.png expected for the Check-in 1 comparison. '
    f'Expected {EXPECTED_SHA256}, got {actual_sha256}'
)

with Image.open(INPUT_PATH) as image:
    assert image.mode == 'RGB', f'Expected RGB PNG, got {image.mode}'
    assert image.size == (512, 512), f'Expected the shared 512x512 crop, got {image.size}'
print('Using:', INPUT_PATH, '|', Image.open(INPUT_PATH).size, '| sha256:', actual_sha256)

## 3. Fixed Check-in 1 settings

`PROMPT` and `SEED` come unchanged from `config/experiment.json`. `Left Light` is the lighting-preference initial latent, not a guarantee of physical sun direction; record it so the result is reproducible. The high-resolution scale is kept at 1.0 so the generated output remains directly comparable at 512×512.

In [ ]:
import json

SETTINGS = {
    'experiment_id': 'berkeley_site01_trial01',
    'view': 'view_000',
    'method': 'ic-light',
    'model': 'lllyasviel/IC-Light text-conditioned; iclight_sd15_fc.safetensors',
    'input': 'data/crops/shattuck_downtown_2025-03/view_000.png',
    'prompt': 'Realistic city street at warm golden hour, natural soft sunlight, preserve original buildings, roads, vehicles and layout',
    'seed': 42,
    'resolution': [512, 512],
    'steps': 25,
    'cfg': 2.0,
    'lighting_preference': 'Left Light',
    'lowres_denoise': 0.90,
    'highres_scale': 1.00,
    'highres_denoise': 0.50,
    'added_prompt': 'best quality',
    'negative_prompt': 'lowres, bad anatomy, bad hands, cropped, worst quality',
    'preprocessing': 'none; calls official process(), not process_relight()',
}
OUTPUT_DIR = Path('/content/ic_light_chronos_view')
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / 'settings.json').write_text(json.dumps(SETTINGS, indent=2) + '\n')
print(json.dumps(SETTINGS, indent=2))

## 4. Load the official model and run the primary relighting attempt

The official demo automatically downloads `iclight_sd15_fc.safetensors` on its first load. This cell evaluates its model setup but excludes only the Gradio UI block and the unused BRIA segmenter. No project code or model code is forked.

In [ ]:
import gc
import os
import numpy as np
import torch
from PIL import Image

# Permit a safe retry after a partially loaded model without restarting or re-uploading.
for _name in ('tokenizer', 'text_encoder', 'vae', 'unet', 'rmbg', 't2i_pipe', 'i2i_pipe',
              'process', 'process_relight', 'encode_prompt_inner', 'encode_prompt_pair',
              'unet_original_forward'):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

# IC-Light uses PyTorch only; prevent Transformers from importing Colab's TensorFlow.
os.environ['USE_TF'] = '0'

official_source = Path('gradio_demo.py').read_text()
model_source = official_source.split('block = gr.Blocks().queue()', 1)[0]
# The notebook calls process() directly, so omit the unused UI dependency.
model_source = model_source.replace('import gradio as gr\n', '')
# process() never uses the portrait-oriented segmenter; prevent its unnecessary model download.
model_source = model_source.replace(
    'rmbg = BriaRMBG.from_pretrained("briaai/RMBG-1.4")', 'rmbg = None'
)
model_source = model_source.replace(
    'rmbg = rmbg.to(device=device, dtype=torch.float32)', '# rmbg is intentionally unused'
)
# The legacy default low-memory loader can yield an empty UNet on current Colab.
model_source = model_source.replace(
    'UNet2DConditionModel.from_pretrained(sd15_name, subfolder="unet")',
    'UNet2DConditionModel.from_pretrained(sd15_name, subfolder="unet", low_cpu_mem_usage=False)'
)
exec(compile(model_source, 'gradio_demo.py', 'exec'), globals())

input_rgb = np.asarray(Image.open(INPUT_PATH).convert('RGB'))
result = process(
    input_fg=input_rgb,
    prompt=SETTINGS['prompt'],
    image_width=512, image_height=512, num_samples=1,
    seed=SETTINGS['seed'], steps=SETTINGS['steps'],
    a_prompt=SETTINGS['added_prompt'], n_prompt=SETTINGS['negative_prompt'],
    cfg=SETTINGS['cfg'], highres_scale=SETTINGS['highres_scale'],
    highres_denoise=SETTINGS['highres_denoise'],
    lowres_denoise=SETTINGS['lowres_denoise'],
    bg_source=SETTINGS['lighting_preference'],
)[0]

RESULT_PATH = OUTPUT_DIR / 'view_000_seed42_left.png'
assert result.shape[:2] == (512, 512), f'Expected a 512x512 comparison output, got {result.shape[:2]}'
Image.fromarray(result).save(RESULT_PATH)
print('Saved:', RESULT_PATH)

## 5. Save a before/after panel and evaluation record

Inspect façade edges, windows, road markings, lane geometry, all vehicles, bus-stop structures, poles, and signs at full size. Do not report this method as structure-preserving if any object is invented, removed, warped, or moved.

In [ ]:
from IPython.display import display
import csv

before = Image.open(INPUT_PATH).convert('RGB')
after = Image.open(RESULT_PATH).convert('RGB')
# Avoid font rendering: Colab's Pillow build can be internally mismatched after installs.
# Image order is fixed: original on the left, IC-Light result on the right.
panel = Image.new('RGB', (1024, 512), 'white')
panel.paste(before, (0, 0))
panel.paste(after, (512, 0))
PANEL_PATH = OUTPUT_DIR / 'comparison_view_000_seed42_left.png'
panel.save(PANEL_PATH)

findings = '''# M3 IC-Light evaluation — fill after visual review

- Buildings/facades/windows:
- Roads/lane markings/curbs:
- Vehicles (count, shape, placement):
- Street furniture/signs/poles/trees:
- Lighting change achieved:
- Hallucinations or structural distortions:
- Verdict: acceptable only if scene structure is substantially unchanged.
'''
(OUTPUT_DIR / 'findings.md').write_text(findings)

log_row = {
    'image_id': 'shattuck_downtown_2025-03', 'view': 'view_000', 'method': 'ic-light',
    'model_version': SETTINGS['model'], 'prompt': SETTINGS['prompt'], 'seed': 42,
    'denoising_strength': 'lowres=0.90; highres=0.50', 'controlnet_scale': '',
    'device': torch.cuda.get_device_name(0),
    'result_path': 'outputs/ic_light/shattuck_downtown_2025-03/view_000_seed42_left.png',
    'notes': 'Primary text-conditioned run; foreground masking bypassed. Complete visual findings before merging.'
}
log_path = OUTPUT_DIR / 'experiment_log_row.csv'
with log_path.open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=log_row.keys())
    writer.writeheader(); writer.writerow(log_row)
print('Review and copy this verified row into docs/experiment_log.csv:')
print(log_path.read_text())
display(panel)

## 6. Download permitted local evidence

Keep these artifacts outside Git unless the team confirms they are permitted to publish. Copy the CSV row into `docs/experiment_log.csv` only after checking the output and completing `findings.md`.

In [ ]:
from google.colab import files
for path in (RESULT_PATH, PANEL_PATH, OUTPUT_DIR / 'settings.json', OUTPUT_DIR / 'findings.md', OUTPUT_DIR / 'experiment_log_row.csv'):
    files.download(path)